In [5]:
import math

states = ['E', '5', 'I']

transitionProbs = {
    'Start': {'E': 1.0},
    'E': {'E': 0.9, '5': 0.1},
    '5': {'I': 1.0},
    'I': {'I': 0.9, 'End': 0.1}
}

emissionProbs = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},
    '5': {'A': 0.05, 'C': 0.0,  'G': 0.95, 'T': 0.0},
    'I': {'A': 0.4,  'C': 0.1,  'G': 0.1,  'T': 0.4}
}

# Function to Calculate Probability of a Given Path

In [18]:
def get_log_prob_of_a_given_path(path, seq):
    if len(path) != len(seq):
        print("Path and sequence lengths must match")
        return

    logProb = 0.0
    prevState = 'Start'
    
    for state, base in zip(path, seq):
        logProb += math.log(transitionProbs[prevState][state])
        logProb += math.log(emissionProbs[state][base])      
        prevState = state

    logProb += math.log(transitionProbs['I']['End'])
    
    return logProb

# Nature Primer example:
path = "EEEEEEEEEEEEEEEEEE5IIIIIII"
seq = "CTTCATGTGAAAGCAGACGTAAGTCA"

logProb = get_log_prob_of_a_given_path(path, seq)
print("Log probability:", logProb)


Log probability: -41.21967768602254


# Viterbi Algorithm Inplementation

In [55]:
def ViterbiAlgorithm(seq):
    n = len(seq)
    currProb = {}  # currProb[s][p] = best log-prob to reach state s at position p
    backtrackMatrix = {}  # backtrackMatrix[s][p] = previous state used to reach state s at position p
    
    for state in states:
        currProb[state] = [-math.inf] * n
        backtrackMatrix[state] = [None] * n
    
    currProb['E'][0] = math.log(transitionProbs['Start']['E']) + math.log(emissionProbs['E'][seq[0]])
    
    for i in range(1, n):
        base = seq[i]
        for currState in states:
            for prevState in states:
                transProb = transitionProbs.get(prevState, {}).get(currState, 0)
                emitProb = emissionProbs.get(currState, {}).get(base, 0)
                if transProb == 0 or emitProb == 0:
                    continue

                prob = currProb[prevState][i-1] + math.log(transProb) + math.log(emitProb)
                if prob > currProb[currState][i]:
                    currProb[currState][i] = prob
                    backtrackMatrix[currState][i] = prevState


    maxFinalProb = -math.inf
    lastState = None
    for state in states:
        if transitionProbs[state].get('End', 0) != 0:
            finalProb = currProb[state][n-1] + math.log(transitionProbs[state]['End'])

            if finalProb > maxFinalProb:
                maxFinalProb = finalProb
                lastState = state

    path = [lastState]
    for i in range(n-1, 0, -1):
        path.append(backtrackMatrix[path[-1]][i])
    path.reverse()

    return ''.join(path), maxFinalProb

# Nature Primer example:
seq = "CTTCATGTGAAAGCAGACGTAAGTCA"
maxProbPath, logProb = ViterbiAlgorithm(seq)

print("Most likely path:", maxProbPath)
print("Log probability for this path:", logProb)

Most likely path: EEEEEEEEEEEEEEEEEE5IIIIIII
Log probability for this path: -41.21967768602254
